# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data -> check for leakage -> chronological-split baseline models -> report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.isna().sum().sum(), 'missing;', df.asset_tag.nunique(), 'unique assets;', df.duplicated().sum(), 'duplicates')
print('base rate', df.failed_90d.mean().round(4), '| range', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## Leakage check: `parts_replaced`
Per the data dictionary, this is the number of parts replaced on the work order that **closed** the inspection record. It is recorded after the inspection, plausibly because of a failure, so it would not be known at prediction time.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2 or more parts replaced failed, and none with 0 did. That is a near-perfect proxy for the label, so it is **excluded** from the real model. We fit it once below only to show how misleading the score would be.

## Other features and time

In [ ]:
feats = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 5, figsize=(18, 3))
for a, f in zip(ax, feats):
    r = df.groupby(pd.qcut(df[f], 10, duplicates='drop'), observed=True).failed_90d.mean()
    a.plot(range(len(r)), r.values, marker='o'); a.set_title(f + ' (decile)'); a.set_ylim(0, None)
plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.mean().round(3).to_dict())
df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'size'])

The failure rate swings by quarter and the last quarters are low. Part of that may be **right-censoring** (failures after the extract date can't be seen for late inspections), but that doesn't explain the high 2024Q2-Q3 rate, so there is some drift too. We therefore use a **chronological split** (train on the earliest 70%, test on the latest 30%) instead of a random one. Each asset appears once, so there is no asset-level leakage across rows.

In [ ]:
cut = int(len(df) * .7); tr, te = df.iloc[:cut], df.iloc[cut:]
print('train ends', tr.inspected_at.max(), 'rate', tr.failed_90d.mean().round(3), '| test starts', te.inspected_at.min(), 'rate', te.failed_90d.mean().round(3))

def prep(d, cols):
    X = d[cols].copy()
    return pd.get_dummies(X, columns=['site'], dtype=float) if 'site' in X else X

def run(cols, name):
    Xtr = prep(tr, cols); Xte = prep(te, cols).reindex(columns=Xtr.columns, fill_value=0)
    out = {}
    models = {'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
              'hgb': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0)}
    for n, m in models.items():
        m.fit(Xtr, tr.failed_90d); p = m.predict_proba(Xte)[:, 1]
        out[n] = dict(model=m, p=p, auc=roc_auc_score(te.failed_90d, p), ap=average_precision_score(te.failed_90d, p))
    print(name, {k: (round(v['auc'], 3), round(v['ap'], 3)) for k, v in out.items()}, '(AUC, AP); AP of random =', round(te.failed_90d.mean(), 3))
    return out, Xte

base = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct', 'site']
leaky, _ = run(base + ['parts_replaced'], 'WITH parts_replaced (leaky, do not trust)')
honest, Xte = run(base, 'WITHOUT parts_replaced (honest)')

## Is the honest signal real? Bootstrap CI, importance, stability over time

In [ ]:
rng = np.random.default_rng(0); y = te.failed_90d.values; p = honest['hgb']['p']
bs = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(500))]
print('HGB test AUC 95% CI', np.percentile(bs, [2.5, 97.5]).round(3))
pi = permutation_importance(honest['hgb']['model'], Xte, y, scoring='roc_auc', n_repeats=10, random_state=0)
print(pd.Series(pi.importances_mean, Xte.columns).sort_values(ascending=False).round(4))
# univariate AUC of each feature per quarter: 0.5 = no signal, <0.5 = reversed direction
df['q'] = df.inspected_at.dt.to_period('Q').astype(str)
pd.DataFrame({q: {f: roc_auc_score(g.failed_90d, g[f]) for f in feats} for q, g in df.groupby('q')}).T.round(3)

The honest model has real but modest signal (test AUC about 0.76, with a wide interval). `load_pct` and `bearing_temp_c` carry most of it: `load_pct` is monotone (about 6% failures at <=50% load, about 24% above 90%) and its signal strengthens over time. `vibration_rms` is informative early (AUC about 0.78 in 2024Q1) but fades to nothing by 2025Q3-Q4, which is why it gets negative permutation importance on the test period. `ambient_temp_c` and `meter_hours` flip direction from quarter to quarter, so treat them as noise. The relationships are **not stationary**, so a model trained on the past may not hold up.

In [ ]:
t = pd.DataFrame({'y': y, 'p': p}); t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=False)
print(t.groupby('decile').y.agg(['mean', 'size']).round(3).tail(3)); print('test base rate', y.mean().round(3))

## Conclusions
See the summary alongside this notebook. Caveats:
- `parts_replaced` is post-inspection information; a model using it is not a valid predictor.
- Quarter-to-quarter drift and possible right-censoring of late inspections make absolute probabilities unreliable; judge rank-ordering.
- One row per asset, so no per-asset history features were possible.